In [1]:
# ── Cell 1: Imports, DAVID function, DEG extraction, Entrez conversion ─────────
import requests
import pandas as pd
import numpy as np
from io import StringIO
import urllib3
import re
import time
import mygene
import textwrap
import plotly.express as px
import plotly.graph_objects as go

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://davidbioinformatics.nih.gov"

def query_david(entrez_ids, annot="GOTERM_BP_FAT,GOTERM_MF_FAT,GOTERM_CC_FAT,KEGG_PATHWAY"):
    session = requests.Session()
    session.verify = False
    resp1 = session.get(
        f"{BASE_URL}/api.jsp",
        params={"type": "ENTREZ_GENE_ID", "ids": ",".join(entrez_ids),
                "tool": "chartReport", "annot": annot, "count": 1},
        timeout=30
    )
    resp1.raise_for_status()
    rowids_match = re.search(r'rowids\.value="([^"]+)"', resp1.text)
    annot_match  = re.search(r'annot\.value="([^"]+)"',  resp1.text)
    if not rowids_match or not annot_match:
        print("  Could not parse form values")
        return pd.DataFrame()
    resp2 = session.post(
        f"{BASE_URL}/chartReport.jsp",
        data={"rowids": rowids_match.group(1), "annot": annot_match.group(1)},
        timeout=30
    )
    resp2.raise_for_status()
    download_match = re.search(r'href="(data/download/chart_[^"]+\.txt)"', resp2.text)
    if not download_match:
        print("  No download link found in response")
        return pd.DataFrame()
    resp3 = session.get(f"{BASE_URL}/{download_match.group(1)}", timeout=30)
    resp3.raise_for_status()
    try:
        return pd.read_csv(StringIO(resp3.text), sep="\t")
    except Exception as e:
        print(f"  Parse error: {e}")
        return pd.DataFrame()

# ── Load DESeq2 results and extract DEGs ──────────────────────────────────────
combined = pd.read_csv("data/deseq2_results.csv")
df = combined.reset_index() if combined.index.name == 'gene_id' else combined.copy()
sig = df[(df['padj'] < 0.1) & (df['baseMean'] > 0)].copy()
sig['gene_id_clean'] = sig['gene_id'].str.split('.').str[0]
sig = sig[sig['gene_id_clean'].str.startswith('ENSG')]

degs_per_treatment = {
    trt: group['gene_id_clean'].tolist()
    for trt, group in sig.groupby('treatment')
}
print("DEGs per treatment:")
for trt, genes in degs_per_treatment.items():
    print(f"  {trt}: {len(genes)} genes")

# ── Convert ENSG → Entrez IDs ─────────────────────────────────────────────────
mg = mygene.MyGeneInfo()
all_ensg = list(set(g for genes in degs_per_treatment.values() for g in genes))
print(f"\nConverting {len(all_ensg)} ENSG IDs to Entrez IDs...")

gene_info = mg.querymany(all_ensg, scopes="ensembl.gene",
                          fields="entrezgene,symbol", species="human")

ensg_to_entrez = {}
ensg_to_symbol = {}
for g in gene_info:
    if 'entrezgene' in g and not g.get('notfound'):
        ensg_to_entrez[g['query']] = str(g['entrezgene'])
        ensg_to_symbol[g['query']] = g.get('symbol', g['query'])

print(f"Successfully mapped: {len(ensg_to_entrez)} / {len(all_ensg)} genes")

degs_entrez = {}
for trt, genes in degs_per_treatment.items():
    entrez = [ensg_to_entrez[g] for g in genes if g in ensg_to_entrez]
    degs_entrez[trt] = entrez
    print(f"  {trt}: {len(genes)} ENSG → {len(entrez)} Entrez IDs")

DEGs per treatment:
  hATF561: 6 genes
  hATF567: 39 genes
  nZF105: 69 genes
  nZF139: 30 genes

Converting 95 ENSG IDs to Entrez IDs...
Successfully mapped: 74 / 95 genes
  hATF561: 6 ENSG → 4 Entrez IDs
  hATF567: 39 ENSG → 30 Entrez IDs
  nZF105: 69 ENSG → 54 Entrez IDs
  nZF139: 30 ENSG → 16 Entrez IDs


In [2]:
# ── Cell 2: Query DAVID, process results ──────────────────────────────────────
all_charts = []

for trt, entrez_ids in degs_entrez.items():
    if len(entrez_ids) == 0:
        print(f"\nSkipping {trt} — no Entrez IDs")
        continue
    print(f"\nQuerying DAVID for {trt} ({len(entrez_ids)} Entrez IDs)...")
    df_trt = query_david(entrez_ids)
    if len(df_trt) > 0:
        df_trt['treatment'] = trt
        all_charts.append(df_trt)
        print(f"  {len(df_trt)} terms returned")
    else:
        print(f"  No results")
    time.sleep(1)

chart_df = pd.concat(all_charts, ignore_index=True)
chart_df['term_clean'] = chart_df['Term'].str.replace(r'^.*?~', '', regex=True)
chart_df['term_clean'] = chart_df['term_clean'].str.replace(r'^hsa\d+:', '', regex=True)
chart_df['-log10(p)']  = -np.log10(chart_df['PValue'].clip(lower=1e-300))
chart_df['gene_ratio'] = chart_df['Count'] / chart_df['List Total']

sig_results = chart_df[chart_df['Benjamini'] < 0.1].sort_values('PValue').reset_index(drop=True)
print(f"\nSignificant terms (Benjamini < 0.1): {len(sig_results)}")
print(sig_results.groupby('treatment')['term_clean'].count())
chart_df.to_csv("data/david_chart_results.csv", index=False)
print("Saved: david_chart_results.csv")


Querying DAVID for hATF561 (4 Entrez IDs)...
  No download link found in response
  No results

Querying DAVID for hATF567 (30 Entrez IDs)...
  62 terms returned

Querying DAVID for nZF105 (54 Entrez IDs)...
  120 terms returned

Querying DAVID for nZF139 (16 Entrez IDs)...
  68 terms returned

Significant terms (Benjamini < 0.1): 8
treatment
hATF567    3
nZF139     5
Name: term_clean, dtype: int64
Saved: david_chart_results.csv


In [3]:
# ── Cell 3: SNHG14 check + off-target analysis ────────────────────────────────
snhg14 = df[df['gene_id'].str.contains('SNHG14|ENSG00000224078', case=False, na=False)]
print("SNHG14 results:")
print(snhg14[['gene_id', 'treatment', 'log2FoldChange', 'pvalue', 'padj', 'baseMean']]
      .sort_values('treatment').to_string(index=False))
print("\nTarget knockdown summary:")
for _, row in snhg14.sort_values('treatment').iterrows():
    direction = "↓ DOWN" if row['log2FoldChange'] < 0 else "↑ UP"
    sig_label = "✓ significant" if row['padj'] < 0.1 else "✗ NOT significant"
    print(f"  {row['treatment']:12s}  log2FC={row['log2FoldChange']:6.2f}  "
          f"padj={row['padj']:.2e}  {direction}  {sig_label}")

SNHG14 results:
       gene_id treatment  log2FoldChange       pvalue         padj    baseMean
Target(SNHG14)   hATF561       -0.563180 3.980300e-08 3.308127e-04 1502.763691
Target(SNHG14)   hATF567       -0.471849 4.007533e-06 3.149113e-03 1592.899197
Target(SNHG14)    nZF105       -0.941133 1.208025e-18 1.679155e-14 1377.072443
Target(SNHG14)    nZF139       -1.033666 1.551767e-20 5.210835e-16 1269.566491

Target knockdown summary:
  hATF561       log2FC= -0.56  padj=3.31e-04  ↓ DOWN  ✓ significant
  hATF567       log2FC= -0.47  padj=3.15e-03  ↓ DOWN  ✓ significant
  nZF105        log2FC= -0.94  padj=1.68e-14  ↓ DOWN  ✓ significant
  nZF139        log2FC= -1.03  padj=5.21e-16  ↓ DOWN  ✓ significant


In [4]:
# ── Cell 4: Cluster summary + bubble plot ─────────────────────────────────────
cluster_map = {
    'defense response to virus':                        'Antiviral / Innate Immune',
    'response to virus':                                'Antiviral / Innate Immune',
    'antiviral innate immune response':                 'Antiviral / Innate Immune',
    'regulation of viral genome replication':           'Antiviral / Innate Immune',
    'negative regulation of viral process':             'Antiviral / Innate Immune',
    'negative regulation of viral genome replication':  'Antiviral / Innate Immune',
    'defense response':                                 'Antiviral / Innate Immune',
    'response to biotic stimulus':                      'Antiviral / Innate Immune',
    'response to external biotic stimulus':             'Antiviral / Innate Immune',
    'response to other organism':                       'Antiviral / Innate Immune',
    'positive regulation of apoptotic process':         'Cell Death / Apoptosis',
    'apoptotic mitochondrial changes':                  'Cell Death / Apoptosis',
    'cell death':                                       'Cell Death / Apoptosis',
    'programmed cell death':                            'Cell Death / Apoptosis',
    'apoptotic process':                                'Cell Death / Apoptosis',
    'regulation of programmed cell death':              'Cell Death / Apoptosis',
    'regulation of apoptotic process':                  'Cell Death / Apoptosis',
    'positive regulation of programmed cell death':     'Cell Death / Apoptosis',
    'response to stress':                               'Stress Response',
    'response to oxidative stress':                     'Stress Response',
    'energy coupled proton transmembrane transport, against electrochemical gradient': 'Mitochondrial / Metabolic',
    'electron transport coupled proton transport':      'Mitochondrial / Metabolic',
}
cluster_colors = {
    'Antiviral / Innate Immune':  '#c0392b',
    'Cell Death / Apoptosis':     '#8e44ad',
    'Stress Response':            '#e67e22',
    'Mitochondrial / Metabolic':  '#27ae60',
    'Other':                      '#7f8c8d'
}

top = (
    chart_df[chart_df['Category'] == 'GOTERM_BP_FAT']
    .sort_values('PValue')
    .groupby('treatment')
    .head(10)
    .copy()
)
top['cluster'] = top['term_clean'].map(cluster_map).fillna('Other')

plot_df = top.copy()

# ── Print cluster summaries ───────────────────────────────────────────────────
sig_threshold = 1.3
print("=" * 65)
print("BIOLOGICAL PROCESS CLUSTERS & SIGNIFICANCE SUMMARY")
print("=" * 65)
for cluster, color in cluster_colors.items():
    cluster_df = plot_df[plot_df['cluster'] == cluster]
    if len(cluster_df) == 0:
        continue
    sig_terms = cluster_df[cluster_df['-log10(p)'] >= sig_threshold]
    print(f"\n{'─' * 65}")
    print(f"{cluster.upper()}")
    print(f"  Terms: {', '.join(cluster_df['term_clean'].unique())}")
    print(f"  Significant at p<0.05: {len(sig_terms)} / {len(cluster_df)} terms")
    if len(sig_terms) > 0:
        for _, row in sig_terms.sort_values('-log10(p)', ascending=False).iterrows():
            stars = '***' if row['-log10(p)'] >= 3 else '**' if row['-log10(p)'] >= 2 else '*'
            print(f"    {stars} {row['term_clean']:50s} "
                  f"p={row['PValue']:.2e}  [{row['treatment']}]")
print(f"\n{'─' * 65}")
print("SIGNIFICANCE KEY:  *** p<0.001   ** p<0.01   * p<0.05")
print("=" * 65)

BIOLOGICAL PROCESS CLUSTERS & SIGNIFICANCE SUMMARY

─────────────────────────────────────────────────────────────────
ANTIVIRAL / INNATE IMMUNE
  Terms: defense response to virus, response to virus, antiviral innate immune response, response to other organism, response to external biotic stimulus, response to biotic stimulus, defense response, negative regulation of viral genome replication
  Significant at p<0.05: 13 / 13 terms
    *** defense response to virus                          p=1.24e-06  [nZF139]
    *** response to virus                                  p=6.12e-06  [nZF139]
    *** defense response to virus                          p=5.33e-05  [hATF567]
    *** antiviral innate immune response                   p=1.47e-04  [hATF567]
    *** response to virus                                  p=2.46e-04  [hATF567]
    *** antiviral innate immune response                   p=9.20e-04  [nZF105]
    ** antiviral innate immune response                   p=1.18e-03  [nZF139]
    *

In [5]:

# ── Bubble plot ───────────────────────────────────────────────────────────────
def wrap_term(term, width=50):
    return '<br>'.join(textwrap.wrap(term, width=width))

plot_df['term_wrapped'] = plot_df['term_clean'].apply(wrap_term)

label_map = {
    'hATF561': 'hATF561<br><b>6</b>',
    'hATF567': 'hATF567<br><b>39</b>',
    'nZF105':  'nZF105<br><b>69</b>',
    'nZF139':  'nZF139<br><b>30</b>'
}
plot_df['treatment_label'] = plot_df['treatment'].map(label_map)
x_order = [label_map[t] for t in ['hATF561', 'hATF567', 'nZF105', 'nZF139']]

fig = px.scatter(
    plot_df,
    x           = 'treatment_label',
    y           = 'term_wrapped',
    size        = 'gene_ratio',
    color       = '-log10(p)',
    color_continuous_scale = 'RdYlBu_r',
    range_color = [0, plot_df['-log10(p)'].max()],
    hover_data  = {
        'Count':           True,
        'PValue':          ':.2e',
        'Benjamini':       ':.2e',
        'Fold Enrichment': ':.2f',
        'Genes':           True,
        'gene_ratio':      ':.3f',
        'cluster':         True,
        'treatment':       True,
        'term_clean':      True,
        'term_wrapped':    False,
        'treatment_label': False,
        '-log10(p)':       ':.2f'
    },
    title  = 'Biological Processes Activated by Treatment<br>'
             '<sup>Bubble size = % of DEGs in process | '
             'Red = highly significant (p&lt;0.001) | '
             'Blue = marginal (p&lt;0.05)</sup>',
    labels = {
        'treatment_label': 'Treatments — <b>Number of off-target genes</b>',
        'term_wrapped':    '',
        '-log10(p)':       'Significance'
    },
    category_orders = {'treatment_label': x_order}
)

fig.add_trace(go.Scatter(
    x=[ label_map['hATF561']], y=[plot_df['term_wrapped'].iloc[0]],
    mode='markers', marker=dict(size=0, opacity=0, color='white'),
    showlegend=False, hoverinfo='skip'
))
fig.add_vrect(
    x0='-0.45', x1='0.45',
    fillcolor='green', opacity=0.06,
    layer='below', line_width=2, line_color='green'
)
fig.update_layout(
    height = 1150, width = 1000,
    title_font = dict(size=20),
    yaxis  = dict(categoryorder='total ascending', tickfont=dict(size=13), dtick=1),
    xaxis  = dict(title='Treatments — <b>Number of off-target genes</b>',
                  title_font=dict(size=14), tickfont=dict(size=13)),
    margin = dict(l=320, r=200, t=120, b=120),
    coloraxis_colorbar = dict(
        title    = 'Significance<br>(-log10 p)',
        tickvals = [1.3, 2.0, 3.0, 4.0, 5.0],
        ticktext = ['1.3 (p=0.05)', '2.0 (p=0.01)', '3.0 (p=0.001)', '4.0', '5.0'],
        tickfont = dict(size=11)
    )
)
fig.show()
fig.write_image("figures/david_dotplot.png")